**PROYECTO FINAL TRIPLETEN**

In [1]:
!pip install catboost

In [3]:
!pip install lightgbm

In [1]:

### Importación de las librerías para realizar el proyecto ()


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as st
from functools import reduce

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
import lightgbm as lgbm
from lightgbm import LGBMClassifier


from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score

### Montaje del Drive en Colab para traer las carpetas que vamos a utilizar

from google.colab import drive
drive.mount('/content/drive')

### Importación de los archivos que se van a utilizar y asignación de las variables para cada uno

contracts_data = pd.read_csv('/content/drive/MyDrive/TripleTen/MatSprint19_(ProyectoFinal)/final_provider/contract.csv')
info_data = pd.read_csv('/content/drive/MyDrive/TripleTen/MatSprint19_(ProyectoFinal)/final_provider/personal.csv')
internet_data = pd.read_csv('/content/drive/MyDrive/TripleTen/MatSprint19_(ProyectoFinal)/final_provider/internet.csv')
phone_data = pd.read_csv('/content/drive/MyDrive/TripleTen/MatSprint19_(ProyectoFinal)/final_provider/phone.csv')

In [2]:
contracts_data = pd.read_csv(r'C:\Users\AngelXHP\Git_Projects/internet_company_churn_prediction/MatSprint19_(ProyectoFinal)/final_provider/contract.csv')
info_data = pd.read_csv(r'C:\Users\AngelXHP\Git_Projects/internet_company_churn_prediction/MatSprint19_(ProyectoFinal)/final_provider/personal.csv')
internet_data = pd.read_csv(r'C:\Users\AngelXHP\Git_Projects/internet_company_churn_prediction/MatSprint19_(ProyectoFinal)/final_provider/internet.csv')
phone_data = pd.read_csv(r'C:\Users\AngelXHP\Git_Projects/internet_company_churn_prediction/MatSprint19_(ProyectoFinal)/final_provider/phone.csv')

In [3]:
### Se realiza el primer vistazo de los datos para ver la tipología de datos, si hay que realizar correcciones, completar o eliminar datos.

def first_lookup(dataframe):
    print('INFORMACIÓN GENERAL DATAFRAME ////////')
    print(dataframe.info(show_counts = True))
    print()
    print('PRIMERAS FILAS ////////')
    print(dataframe.head())
    print()
    print('ELEMENTOS DUPLICADOS ////////')
    print(dataframe.duplicated().sum())
    print()
    print('VALORES NULOS ////////')
    print(dataframe.isna().sum())

first_lookup(contracts_data)
print('-------------------------------------------------------------------------------------')
print('\n')
first_lookup(info_data)
print('-------------------------------------------------------------------------------------')
print('\n')
first_lookup(internet_data)
print('-------------------------------------------------------------------------------------')
print('\n')
first_lookup(phone_data)

INFORMACIÓN GENERAL DATAFRAME ////////
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   BeginDate         7043 non-null   object 
 2   EndDate           7043 non-null   object 
 3   Type              7043 non-null   object 
 4   PaperlessBilling  7043 non-null   object 
 5   PaymentMethod     7043 non-null   object 
 6   MonthlyCharges    7043 non-null   float64
 7   TotalCharges      7043 non-null   object 
dtypes: float64(1), object(7)
memory usage: 440.3+ KB
None

PRIMERAS FILAS ////////
   customerID   BeginDate              EndDate            Type  \
0  7590-VHVEG  2020-01-01                   No  Month-to-month   
1  5575-GNVDE  2017-04-01                   No        One year   
2  3668-QPYBK  2019-10-01  2019-12-01 00:00:00  Month-to-month   
3  7795-CFOCW  2016-05-01            

In [4]:
### Se realiza la unión de los 4 dataframes para validar cuales son los datos faltantes en cada uno de ellos y hacer el data-wrangling
### que sea necesario, se visualizan los datos nuevamente con la función "first_lookup".

dfs_wom = [info_data, contracts_data, internet_data, phone_data]

merged_dataframes = reduce(lambda left, right: pd.merge(left, right, on='customerID', how='outer'), dfs_wom)
first_lookup(merged_dataframes)

INFORMACIÓN GENERAL DATAFRAME ////////
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   BeginDate         7043 non-null   object 
 6   EndDate           7043 non-null   object 
 7   Type              7043 non-null   object 
 8   PaperlessBilling  7043 non-null   object 
 9   PaymentMethod     7043 non-null   object 
 10  MonthlyCharges    7043 non-null   float64
 11  TotalCharges      7043 non-null   object 
 12  InternetService   5517 non-null   object 
 13  OnlineSecurity    5517 non-null   object 
 14  OnlineBackup      5517 non-null   object 
 15  DeviceProtection  5517 non-null   object 
 16  Tec

In [5]:
### Para evitar eliminaciones de las filas con datos incompletos, realizo rellenado de las celdas con 'Unknown', ya que equivalían
### a más del 10% de los datos e iba a afectar la proporción de los mismos para realizar el modelado, se valida nuevamente el dataframe.

filled_merged_dataframes = merged_dataframes.fillna('Unknown')

filled_merged_dataframes_1 = filled_merged_dataframes.copy()
filled_merged_dataframes_1.isna().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
BeginDate           0
EndDate             0
Type                0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
MultipleLines       0
dtype: int64

INICIO PRUEBA ·······································

In [11]:
import os
os.chdir('C:/Users/AngelXHP/Git_Projects/internet_company_churn_prediction/')
print(os.getcwd())

C:\Users\AngelXHP\Git_Projects\internet_company_churn_prediction


In [13]:

import sys
import pytest
sys.path.append("..")

from src.preprocess import (
    validate_columns,
    clean_data,
    create_features,
    preprocess_data,
    prepare_features,
)



In [25]:
model, metrics = train_model(filled_merged_dataframes)

C:\Users\AngelXHP\Git_Projects\internet_company_churn_prediction\src\preprocess.py:47: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["EndDate"] = pd.to_datetime(


In [19]:
clientes_prueba = filled_merged_dataframes.head(5).copy()
clientes_prueba = clientes_prueba.drop(
    columns="EndDate",
    errors="ignore"
)
clientes_prueba = clientes_prueba[["gender", "SeniorCitizen"]]
clientes_prueba

,gender,SeniorCitizen
0,Female,0
1,Male,0
2,Male,0
3,Male,1
4,Female,1


In [25]:
resultado = validate_columns(clientes_prueba, ["gender", "SeniorCitzen"])

resultado

ValueError: Faltan columnas requeridas: ['SeniorCitzen']

In [25]:
print(df_processed.columns.tolist())

['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'BeginDate', 'EndDate', 'Type', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'MultipleLines', 'Churn', 'TenureMonths', 'NumServices', 'HasInternet', 'HasStreaming', 'SecurityPack']


In [27]:
columnas_notebook = set(filled_merged_dataframes_1.columns)
columnas_preprocess = set(df_processed.columns)

print("Columnas solo en el notebook:")
print(columnas_notebook - columnas_preprocess)

print("\nColumnas solo en preprocess.py:")
print(columnas_preprocess - columnas_notebook)

Columnas solo en el notebook:
set()

Columnas solo en preprocess.py:
{'NumServices', 'SecurityPack', 'Churn', 'HasInternet', 'HasStreaming', 'TenureMonths'}


In [37]:
X, y = prepare_features(df_processed)

In [39]:
print(X.columns.tolist())

['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'Type', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'MultipleLines', 'TenureMonths', 'NumServices', 'HasInternet', 'HasStreaming', 'SecurityPack']


In [49]:
model, metrics = train_model(filled_merged_dataframes_1)

metrics

C:\Users\AngelXHP\Git_Projects\internet_company_churn_prediction\src\preprocess.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["EndDate"] = pd.to_datetime(


{'accuracy': 0.902328222600795,
 'precision': 0.8624078624078624,
 'recall': 0.7516059957173448,
 'f1': 0.8032036613272311,
 'roc_auc': 0.9409728312852268}

FIN DE PRUEBA --------------------------------

In [ ]:
### Conversión de columnas "TotalCharges", "BeginDate" y "EndDate" a sus respectivos tipos de datos para poder trabajar con ellos más adelante.

filled_merged_dataframes_1["TotalCharges"] = pd.to_numeric(filled_merged_dataframes_1["TotalCharges"], errors="coerce")
filled_merged_dataframes_1["BeginDate"] = pd.to_datetime(filled_merged_dataframes_1["BeginDate"])
filled_merged_dataframes_1["EndDate"] = pd.to_datetime(filled_merged_dataframes_1["EndDate"], errors="coerce")

### Fijamos la fecha límite de nuestros datos para obtener las columnas referentes al tiempo de permanencia de los usuarios y al abandono del servicio.

fecha_corte = pd.Timestamp("2020-02-01")
filled_merged_dataframes_1["Churn"] = filled_merged_dataframes_1["EndDate"].notna().astype(int)
filled_merged_dataframes_1["TenureMonths"] = ((fecha_corte - filled_merged_dataframes_1["BeginDate"]).dt.days / 30.44).round(0).astype(int)

### Realizo ingeniería de características de las que considero más relevantes, tomando en cuenta el número de servicios con los que cuentan los usuarios, así como su gasto promedio mensual.

servicios = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

filled_merged_dataframes_1["NumServices"] = (filled_merged_dataframes_1[servicios] == "Yes").sum(axis=1)
filled_merged_dataframes_1["HasInternet"] = (filled_merged_dataframes_1["InternetService"] != "No").astype(int)
filled_merged_dataframes_1["HasStreaming"] = ((filled_merged_dataframes_1["StreamingTV"] == "Yes") |(filled_merged_dataframes_1["StreamingMovies"] == "Yes")).astype(int)
filled_merged_dataframes_1["SecurityPack"] = ((filled_merged_dataframes_1["OnlineSecurity"] == "Yes") |(filled_merged_dataframes_1["DeviceProtection"] == "Yes")).astype(int)
filled_merged_dataframes_1["TotalCharges"] = filled_merged_dataframes_1["TotalCharges"].fillna(0)

### Se realiza una previa revisión estadística de los elementos númericos del dataframe para analizar qué tipo de gráficos y sobre qué observaciones se realizarán.

filled_merged_dataframes_1.describe()



In [ ]:
### Se visualiza previamente el porcentaje actual de cancelación general de los clientes, que también nos ayudará a validar el balanceo de clases.

ax = sns.countplot(data=filled_merged_dataframes_1, x="Churn", hue="Churn", legend=False)
px = filled_merged_dataframes_1["Churn"].value_counts(normalize=True)
etiquetas = [f"{idx} ({val:.1%})" for idx, val in px.items()]
plt.legend(handles=ax.patches, labels=etiquetas, title="Porcentaje")
plt.title("Porcentaje de cancelación general")
ax.set_xticks([0, 1], labels=["No Cancelado", "Cancelado"])
plt.show()

In [ ]:
### Se visualiza también la cancelación por duración del servicio, según el tipo de contrato del cliente.

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

counts_text = str(filled_merged_dataframes_1["Type"].value_counts())
ax[0].text(0.1, 0.5, counts_text, fontsize=12, family="monospace", va="center")
ax[0].axis("off")

sns.countplot(data=filled_merged_dataframes_1, x="Type", hue="Churn", ax=ax[1])
ax[1].set_title("Cancelación por duración de contrato")
plt.legend(labels=['No cancelado', 'Cancelado'])
plt.tight_layout()
plt.show()

In [ ]:
### Determinamos cuales serán las características númericas y categóricas.

numerical_features = ["MonthlyCharges", "TotalCharges", "TenureMonths", "NumServices"]

categorical_features = ["InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies", "Type",
    "PaperlessBilling", "PaymentMethod", "gender", "SeniorCitizen", "Partner", "Dependents", "MultipleLines"]

### Realizamos la primera separación de datos que utilizaremos para los modelos que no necesitan transformación, ni balanceo.

features = filled_merged_dataframes_1.drop(columns=["customerID", "BeginDate", "EndDate", "Churn"])
target = filled_merged_dataframes_1["Churn"]

features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.25, stratify=target, random_state=12345)

In [ ]:
### Se modela el baseline a partir de un clasificador Dummy con el objetivo de visualizar si hay o no mejoría al presentar los demás modelos de clasificación.
### Se obtienen las primeras métricas de evaluación del modelo.

dummy = DummyClassifier(
    strategy="most_frequent",
    random_state=12345
)

dummy.fit(features_train, target_train)
dummy_pred = dummy.predict(features_test)
dummy_prob = dummy.predict_proba(features_test)[:,1]

print("Accuracy :", accuracy_score(target_test, dummy_pred))
print("Precision:", precision_score(target_test, dummy_pred, zero_division=0))
print("Recall   :", recall_score(target_test, dummy_pred))
print("F1 Score :", f1_score(target_test, dummy_pred))
print("ROC AUC:", roc_auc_score(target_test, dummy_prob))


In [ ]:
### Escalado, codificado y entrenado para modelo de regresión logística, encadenado dentro de un pipeline.

preprocessor_lr = ColumnTransformer(transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)])

pipeline_lr = Pipeline([
    ("preprocessor", preprocessor_lr),
    ("classifier", LogisticRegression(class_weight="balanced"))])

pipeline_lr.fit(features_train, target_train)

In [ ]:
### Se obtienen los primeros resultados de predicción para regreson logística con sus respectivas métricas de evaluación.

log_pred = pipeline_lr.predict(features_test)
log_prob = pipeline_lr.predict_proba(features_test)[:,1]

print("Accuracy :", accuracy_score(target_test, log_pred))
print("Precision:", precision_score(target_test, log_pred))
print("Recall   :", recall_score(target_test, log_pred))
print("F1 Score :", f1_score(target_test, log_pred))
print("ROC AUC:", roc_auc_score(target_test, log_prob))

In [ ]:
### Se observa la primera matriz de confusión que nos ayuda a evaluar el rendimiento de las predicciones del modelo de regresión logística

ConfusionMatrixDisplay.from_predictions(
    target_test,
    log_pred,
    cmap="Blues")
plt.show()

In [ ]:
### Codificado y entrenado para modelo de árbol de decisión, encadenado dentro de un pipeline.

preprocessor_dt = ColumnTransformer(transformers=[
        ("num", "passthrough", numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)])

pipeline_dt = Pipeline([
    ("preprocessor", preprocessor_dt),
    ("classifier", DecisionTreeClassifier(max_depth=5, min_samples_split=20, min_samples_leaf=10, class_weight="balanced", random_state=12345))])

pipeline_dt.fit(features_train, target_train)

In [ ]:
### Se obtienen los resultados de predicción para el árbol de decisión con sus respectivas métricas de evaluación.

dt_pred = pipeline_dt.predict(features_test)
dt_prob = pipeline_dt.predict_proba(features_test)[:,1]

print("Accuracy :", accuracy_score(target_test, dt_pred))
print("Precision:", precision_score(target_test, dt_pred))
print("Recall   :", recall_score(target_test, dt_pred))
print("F1 Score :", f1_score(target_test, dt_pred))
print("ROC AUC:", roc_auc_score(target_test, dt_prob))

In [ ]:
### Se observa la segunda matriz de confusión que nos ayuda a evaluar el rendimiento de las predicciones del modelo de árbol de decisión

ConfusionMatrixDisplay.from_predictions(
    target_test,
    dt_pred,
    cmap="Blues")
plt.show()

In [ ]:
### Codificado y entrenado para modelo de bosque aleatorio, encadenado dentro de un pipeline.

preprocessor_rf = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)])

pipeline_rf = Pipeline([
    ("preprocessor", preprocessor_rf),
    ("classifier", RandomForestClassifier(n_estimators=81, class_weight="balanced", random_state=12345))])

pipeline_rf.fit(features_train, target_train)

In [ ]:
### Se obtienen los resultados de predicción para el bosque aleatorio con sus respectivas métricas de evaluación.

rf_pred = pipeline_rf.predict(features_test)
rf_prob = pipeline_rf.predict_proba(features_test)[:,1]

print("Accuracy :", accuracy_score(target_test, rf_pred))
print("Precision:", precision_score(target_test, rf_pred))
print("Recall   :", recall_score(target_test, rf_pred))
print("F1 Score :", f1_score(target_test, rf_pred))
print("ROC AUC:", roc_auc_score(target_test, rf_prob))

In [ ]:
### Se observa la tercera matriz de confusión que nos ayuda a evaluar el rendimiento de las predicciones del modelo de bosque aleatorio

ConfusionMatrixDisplay.from_predictions(
    target_test,
    rf_pred,
    cmap="Blues")
plt.show()

In [ ]:
### Se ejecuta y entrena modelo CatBoost.

model_catbs = CatBoostClassifier(iterations=500, learning_rate=0.1, cat_features=categorical_features, verbose=100,
                                 loss_function = "Logloss", random_seed=12345, auto_class_weights='Balanced')

model_catbs.fit(features_train, target_train)


In [ ]:
### Se obtienen los resultados de predicción para CatBoost con sus respectivas métricas de evaluación.

cb_pred = model_catbs.predict(features_test)
cb_prob = model_catbs.predict_proba(features_test)[:,1]


print("Accuracy :", accuracy_score(target_test, cb_pred))
print("Precision:", precision_score(target_test, cb_pred))
print("Recall   :", recall_score(target_test, cb_pred))
print("F1 Score :", f1_score(target_test, cb_pred))
print("ROC AUC:", roc_auc_score(target_test,cb_prob))

In [ ]:
### Se observa la cuarta matriz de confusión que nos ayuda a evaluar el rendimiento de las predicciones del modelo CatBoost.

ConfusionMatrixDisplay.from_predictions(
    target_test,
    cb_pred,
    cmap="Blues"
)
plt.show()

In [ ]:
### Obtenemos una copia para ejecutar el modelo LightGBM ya que este funciona con tipos de datos específico.
### Recorremos las características categóricas para cambiar el tipo de dato a categoría
### Entrenamos el modelo LightGBM

features_train_lgbm = features_train.copy()
features_test_lgbm = features_test.copy()

for col in categorical_features:
    features_train_lgbm[col] = features_train_lgbm[col].astype("category")
    features_test_lgbm[col] = features_test_lgbm[col].astype("category")

model_lightgbm = lgbm.LGBMClassifier(objective="binary", n_estimators=500, learning_rate=0.1, random_state=12345, verbose = -1)
model_lightgbm.fit(features_train_lgbm, target_train, eval_metric="auc")

In [ ]:
### Se obtienen los resultados de predicción para LightGBM con sus respectivas métricas de evaluación.

lgbm_pred = model_lightgbm.predict(features_test_lgbm)
lgbm_prob = model_lightgbm.predict_proba(features_test_lgbm)[:,1]


print("Accuracy :", accuracy_score(target_test, lgbm_pred))
print("Precision:", precision_score(target_test, lgbm_pred))
print("Recall   :", recall_score(target_test, lgbm_pred))
print("F1 Score :", f1_score(target_test, lgbm_pred))
print("ROC AUC:", roc_auc_score(target_test, lgbm_prob))

In [ ]:
### Se observa la cuarta matriz de confusión que nos ayuda a evaluar el rendimiento de las predicciones del modelo LightGBM.

ConfusionMatrixDisplay.from_predictions(
    target_test,
    lgbm_pred,
    cmap="Blues")
plt.show()

In [ ]:
### Se guardan los resultados obtenidos de cada modelo para realizar la comparáción de los mismos.

resultados = []

def add_rst(nombre, y_true, y_pred, y_prob):
    resultados.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC_AUC": roc_auc_score(y_true, y_prob)
    })

add_rst("Dummy", target_test, dummy_pred, dummy_prob)
add_rst("Logistic Regression", target_test, log_pred, log_prob)
add_rst("Decision Tree", target_test, dt_pred, dt_prob)
add_rst("Random Forest", target_test, rf_pred, rf_prob)
add_rst("CatBoost", target_test, cb_pred, cb_prob)
add_rst("LightGBM", target_test, lgbm_pred, lgbm_prob)


In [ ]:
### Se visualizan y se comparan los resultados obtenidos por los modelos evaluados.

resultados_df = pd.DataFrame(resultados)

resultados_df = resultados_df.sort_values(by="ROC_AUC", ascending=False).reset_index(drop=True)

resultados_df


**Se escogen los dos mejores modelos, para realizar la validación y búsqueda de hiperparámetros.
En este caso puntual serán: LightGMB y CatBoost**

**MODELO FINAL CATBOOST ⬇**

In [ ]:
### Búsqueda y validación de hiperparámetros para Catboost

parametros_cat = {
    "iterations": [50, 100, 200, 400],
    "depth": [4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "l2_leaf_reg": [1, 3, 5, 7, 10]
}

busqueda_catboost = RandomizedSearchCV(
    estimator=CatBoostClassifier(
        auto_class_weights="Balanced",
        random_seed=12345,
        verbose=0
    ),
    param_distributions=parametros_cat,
    n_iter=20,
    scoring="roc_auc",
    cv=5,
    random_state=12345,
    n_jobs=-1
)

In [ ]:
### Entrenamiento con los parámetros de prueba

busqueda_catboost.fit(
    features_train,
    target_train,
    cat_features=categorical_features
)

In [ ]:
### Obtención de los mejores parámetros para Catboost

mejor_catboost = busqueda_catboost.best_estimator_
mejor_catboost

In [ ]:
### Ejecución y entrenamiento del modelo final de Catboost con los mejores hiperparámetros

final_catboost = CatBoostClassifier(iterations=400, learning_rate=0.1, cat_features=categorical_features, verbose=0, depth=5, l2_leaf_reg=5,
                                 loss_function = "Logloss", random_seed=12345, auto_class_weights='Balanced')
final_catboost.fit(features_train, target_train)

In [ ]:
### Se obtienen los resultados de predicción para el modelo final CatBoost con sus respectivas métricas de evaluación.

fina_cb_pred = final_catboost.predict(features_test)
final_cb_prob = final_catboost.predict_proba(features_test)[:,1]

print("Accuracy :", accuracy_score(target_test, fina_cb_pred))
print("Precision:", precision_score(target_test, fina_cb_pred))
print("Recall   :", recall_score(target_test, fina_cb_pred))
print("F1 Score :", f1_score(target_test, fina_cb_pred))
print("ROC AUC:", roc_auc_score(target_test,final_cb_prob))

add_rst("CatBoost_enhanced", target_test, fina_cb_pred, final_cb_prob)

**MODELO FINAL LIGHTGBM ⬇**

In [ ]:
### Preparación de parámetros de prueba para LGBM

parametros_lgbm = {
    "n_estimators": [300, 500, 700, 1000],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "num_leaves": [15, 31, 50, 75],
    "max_depth": [-1, 4, 6, 8],
    "min_child_samples": [10, 20, 30, 50],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}

In [ ]:
### Modelo base LGBM

lightgbm_base = LGBMClassifier(
    objective="binary",
    random_state=12345,
    n_jobs=1,
    verbosity=-1
)

### Búsqueda y validación de hiperparámetros para LightGBM

random_search_lgbm = RandomizedSearchCV(
    estimator=lightgbm_base,
    param_distributions=parametros_lgbm,
    n_iter=10,
    scoring="roc_auc",
    cv=5,
    random_state=12345,
    n_jobs=-1,
    verbose=1
)

In [ ]:
### Entrenamiento con los parámetros de prueba

random_search_lgbm.fit(
    features_train_lgbm,
    target_train,
    categorical_feature=categorical_features
)

In [ ]:
### Obtención de los mejores parámetros para LightGBM

mejor_lgbm = random_search_lgbm.best_estimator_
mejor_lgbm

In [ ]:
### Ejecución y entrenamiento del modelo final de LightGBM con los mejores hiperparámetros

final_lgbm = LGBMClassifier(colsample_bytree=0.9, max_depth=4, min_child_samples=50,
               n_estimators=700, n_jobs=1, objective='binary',
               random_state=12345, subsample=0.8, verbosity=-1)
final_lgbm.fit(features_train_lgbm, target_train, eval_metric="auc")

In [ ]:
### Se obtienen los resultados de predicción para el modelo final LightGBM con sus respectivas métricas de evaluación.

final_lgbm_pred = final_lgbm.predict(features_test_lgbm)
final_lgbm_prob = final_lgbm.predict_proba(features_test_lgbm)[:,1]

print("Accuracy :", accuracy_score(target_test, final_lgbm_pred))
print("Precision:", precision_score(target_test, final_lgbm_pred))
print("Recall   :", recall_score(target_test, final_lgbm_pred))
print("F1 Score :", f1_score(target_test, final_lgbm_pred))
print("ROC AUC:", roc_auc_score(target_test, final_lgbm_prob))

add_rst("LightGBM_enhanced", target_test, final_lgbm_pred, final_lgbm_prob)

In [ ]:
### Se realiza nuevamente la comparación de modelos de acuerdo a las nuevas métricas obtenidas.

resultados_df = pd.DataFrame(resultados)
resultados_df.sort_values("ROC_AUC", ascending=False)

**Después de la última verificación de los 2 mejores modelos optimizados, se toma la decisión de continuar trabajando con LightGBM a partir de esta línea para la presentación de resultados ⬇**

In [ ]:
### Se realiza la tabulación de la importancia de variables que validó el modelo seleccionado LightGBM para predecir el abandono de clientes

importancias = pd.DataFrame({
    "Variable": features_train.columns,
    "Importancia": final_lgbm.feature_importances_
}).sort_values(
    "Importancia",
    ascending=False
)
importancias.head(15)

**Los clientes con contratos mensuales, menor antigüedad y determinados
servicios presentan una mayor probabilidad de abandono.**

In [ ]:
### Se realiza la visualización de las mismas varibles

top_variables_lgbm = importancias.head(15)

plt.figure(figsize=(9, 6))
plt.barh(
    top_variables_lgbm["Variable"],
    top_variables_lgbm["Importancia"]
)
plt.gca().invert_yaxis()
plt.xlabel("Nivel de importancia")
plt.title("Variables más importantes para predecir churn")
plt.show()

In [ ]:
clientes_riesgo = features_test_lgbm.copy()
clientes_riesgo

In [ ]:
### Se obtiene la tabla de clientes con su respectivo ID, de acuerdo a la probabilidad de abandono que presenta cada uno.

clientes_riesgo["customerID"] = filled_merged_dataframes_1.loc[features_test_lgbm.index, "customerID"]
clientes_riesgo["Churn_real"] = target_test.values
clientes_riesgo["Churn_predicho"] = final_lgbm_pred
clientes_riesgo["Probabilidad_churn"] = final_lgbm_prob

clientes_riesgo

In [ ]:
clientes_riesgo['Estado_actual'] = (
    clientes_riesgo["Churn_real"]
    .map({
        0: "Activo",
        1: "Cancelado"}))

clientes_riesgo['Prediccion_servicio'] = (
    clientes_riesgo['Churn_predicho']
    .map({
        0: "Permanece",
        1: 'Abandono/Riesgo'}))

clientes_riesgo['Prediccion_correcta'] = np.where(
    clientes_riesgo["Churn_real"]
    == clientes_riesgo["Churn_predicho"],
    "Correcta",
    "Incorrecta")



In [ ]:
### Se crea una nueva columna para clasificar su nivel de riesgo de acuerdo a la probabilidad de abandono.

clientes_riesgo["Nivel_riesgo"] = pd.cut(
    clientes_riesgo["Probabilidad_churn"],
    bins=[0, 0.30, 0.50, 0.80, 1.00],
    labels=["Bajo", "Medio", "Alto", "Crítico"],
    include_lowest=True
)


In [ ]:
conditions = [
    (clientes_riesgo["Churn_real"] == 1) &
    (clientes_riesgo["Churn_predicho"] == 1),

    (clientes_riesgo["Churn_real"] == 0) &
    (clientes_riesgo["Churn_predicho"] == 0),

    (clientes_riesgo["Churn_real"] == 0) &
    (clientes_riesgo["Churn_predicho"] == 1),

    (clientes_riesgo["Churn_real"] == 1) &
    (clientes_riesgo["Churn_predicho"] == 0)]

choices = [
    "Verdadero Positivo",
    "Verdadero Negativo",
    "Falso Positivo",
    "Falso Negativo"]

clientes_riesgo["Tipo_Prediccion"] = np.select(
    conditions,
    choices,
    default="Sin clasificar")

clientes_riesgo

In [ ]:
columnas_principales = [
    "customerID",
    "Churn_real",
    "Churn_predicho",
    "Probabilidad_churn",
    "Estado_actual",
    "Prediccion_servicio",
    "Prediccion_correcta",
    "Nivel_riesgo",
    "Tipo_Prediccion"
]

In [ ]:
otras_columnas = [
    col for col in clientes_riesgo.columns
    if col not in columnas_principales]

clientes_riesgo = clientes_riesgo[
    columnas_principales + otras_columnas]

clientes_riesgo

In [ ]:
clientes_riesgo["Probabilidad_churn"].describe()

In [ ]:
### Se exportan las tablas para realizar las visualizaciones en PowerBI.

clientes_riesgo.to_csv(
    "/content/drive/MyDrive/TripleTen/MatSprint19_(ProyectoFinal)/clientes_riesgo_churn.csv",
    index=False
)

resultados_df.to_csv(
    "/content/drive/MyDrive/TripleTen/MatSprint19_(ProyectoFinal)/comparacion_modelos.csv",
    index=False
)

importancias.to_csv(
    "/content/drive/MyDrive/TripleTen/MatSprint19_(ProyectoFinal)/importancia_variables.csv",
    index=False
)